# 🛒 Hybrid Recommender System — Amazon Products
**Mata Kuliah:** Text Mining  
**Metode:** Content-Based Filtering (TF-IDF + Cosine Similarity) + Collaborative Filtering (Item-Based) → Hybrid

---

## Pipeline
```
amazon.csv
   │
   ├──→ [Content-Based]  TF-IDF (review_content + about_product) → Cosine Similarity
   ├──→ [CF]             Explode user_id → User-Item Matrix → Item-Item Similarity
   └──→ [Hybrid]         α × CF + (1-α) × CB → Top-N Rekomendasi
```

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded ✓')

## 2. Load & Inspeksi Dataset

In [ ]:
df = pd.read_csv('amazon.csv')
print(f'Shape: {df.shape}')
print(f'Kolom: {df.columns.tolist()}')
df.head(3)

### 2.1 Temuan Kunci Dataset
| Kolom | Temuan | Penanganan |
|-------|--------|------------|
| `rating` | Dtype `object`, ada nilai `'|'` korup | `pd.to_numeric(errors='coerce')` |
| `user_id` | Multi-value per baris (dipisah koma) | Explode → ~9.042 interaksi |
| `review_content` | Multi-value sesuai jumlah user | Explode sinkron dengan `user_id` |
| `about_product` | Deskripsi produk — kaya konten | Gabung dengan `review_content` untuk CB |

## 3. Data Preparation

In [ ]:
# ── 3.1 Fix rating dtype ──────────────────────────────────────
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])
print(f'Setelah clean rating: {df.shape}')
print(f'Distribusi rating:')
print(df['rating'].describe())

In [ ]:
# ── 3.2 Explode user_id + review_content (sinkron) ───────────
def sync_explode(df_in):
    rows = []
    for _, row in df_in.iterrows():
        uids = row['user_id'].split(',')
        revs = row['review_content'].split(',')
        n = len(uids)
        revs = revs[:n] + ['']*(n - len(revs)) if len(revs) < n else revs[:n]
        for i, uid in enumerate(uids):
            rows.append({
                'product_id': row['product_id'],
                'product_name': row['product_name'],
                'category': row['category'],
                'rating': row['rating'],
                'about_product': row['about_product'],
                'user_id': uid.strip(),
                'review_content': revs[i].strip(),
            })
    return pd.DataFrame(rows)

df_long = sync_explode(df)
print(f'Setelah explode: {df_long.shape}')
print(f'Unique users   : {df_long["user_id"].nunique()}')
print(f'Unique products: {df_long["product_id"].nunique()}')
df_long.head(3)

## 4. Content-Based Filtering (Text Mining)

In [ ]:
# ── 4.1 Bangun product-level DataFrame ───────────────────────
prod_reviews = df_long.groupby('product_id')['review_content'].apply(
    lambda x: ' '.join(x.dropna().astype(str))
).reset_index()

prod_df = df[['product_id','product_name','category','rating','about_product']] \
    .drop_duplicates('product_id').copy()
prod_df = prod_df.merge(prod_reviews, on='product_id', how='left')

# Gabungkan about_product + aggregated reviews
prod_df['text_combined'] = (
    prod_df['about_product'].fillna('') + ' ' +
    prod_df['review_content'].fillna('')
)
print(f'Product-level df: {prod_df.shape}')
print('\nContoh text_combined (100 char pertama):')
print(prod_df['text_combined'].iloc[0][:100])

In [ ]:
# ── 4.2 Text Preprocessing ────────────────────────────────────
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)          # hapus URL
    text = re.sub(r'[^a-z0-9\s]', ' ', text)    # hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip()
    return text

prod_df['text_clean'] = prod_df['text_combined'].apply(preprocess)
print('Preprocessing selesai ✓')
print('Contoh output preprocessing:')
print(prod_df['text_clean'].iloc[0][:150])

In [ ]:
# ── 4.3 TF-IDF Vectorizer ─────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2),   # unigram + bigram
    min_df=2              # abaikan term yang sangat jarang
)

tfidf_matrix = tfidf.fit_transform(prod_df['text_clean'])
print(f'TF-IDF matrix shape : {tfidf_matrix.shape}')
print(f'Sparsity            : {1 - tfidf_matrix.nnz/(tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.4f}')

# Top features
feature_names = tfidf.get_feature_names_out()
print(f'\nSample features: {feature_names[:20].tolist()}')

In [ ]:
# ── 4.4 Cosine Similarity Matrix ──────────────────────────────
cb_sim = cosine_similarity(tfidf_matrix)
print(f'CB similarity matrix: {cb_sim.shape}')
print(f'Min: {cb_sim.min():.4f} | Max (diagonal): {cb_sim.diagonal().mean():.4f}')
print(f'Mean off-diagonal similarity: {(cb_sim.sum() - cb_sim.trace()) / (cb_sim.size - len(cb_sim)):.4f}')

## 5. Collaborative Filtering (Item-Based)

In [ ]:
# ── 5.1 User-Item Matrix ──────────────────────────────────────
user_item = df_long.pivot_table(
    index='user_id', columns='product_id',
    values='rating', aggfunc='mean'
).fillna(0)

print(f'User-item matrix shape : {user_item.shape}')
filled = (user_item > 0).values.sum()
total = user_item.shape[0] * user_item.shape[1]
print(f'Sparsity               : {1 - filled/total:.4f} ({(1-filled/total)*100:.1f}% kosong)')

In [ ]:
# ── 5.2 Item-Item CF Similarity ───────────────────────────────
item_sim_cf = cosine_similarity(user_item.T)
item_sim_cf_df = pd.DataFrame(
    item_sim_cf,
    index=user_item.columns,
    columns=user_item.columns
)
print(f'CF item similarity matrix: {item_sim_cf_df.shape}')
print('\nSample similarity (5x5):')
item_sim_cf_df.iloc[:5, :5].round(3)

## 6. Hybrid Recommender

In [ ]:
ALPHA = 0.4  # bobot CF; (1-ALPHA) = bobot CB

prod_ids = prod_df['product_id'].tolist()
prod_idx = {pid: i for i, pid in enumerate(prod_ids)}

def hybrid_recommend(product_id, top_n=5, alpha=ALPHA):
    """
    Hybrid recommendation:
    score = alpha * CF_score + (1-alpha) * CB_score
    """
    if product_id not in prod_idx:
        return pd.DataFrame()
    idx = prod_idx[product_id]
    
    # CB scores dari TF-IDF cosine similarity
    cb_scores = cb_sim[idx]
    
    # CF scores — align ke urutan prod_ids
    if product_id in item_sim_cf_df.index:
        cf_row = item_sim_cf_df.loc[product_id]
        cf_scores = np.array([cf_row.get(pid, 0) for pid in prod_ids])
    else:
        cf_scores = np.zeros(len(prod_ids))
    
    # Normalisasi ke [0,1]
    cb_norm = cb_scores / (cb_scores.max() + 1e-9)
    cf_norm = cf_scores / (cf_scores.max() + 1e-9)
    
    # Hybrid score
    hybrid = alpha * cf_norm + (1 - alpha) * cb_norm
    hybrid[idx] = -1  # exclude produk itu sendiri
    
    top_idx = np.argsort(hybrid)[::-1][:top_n]
    recs = []
    for i in top_idx:
        pid = prod_ids[i]
        row = prod_df[prod_df['product_id'] == pid].iloc[0]
        recs.append({
            'product_id': pid,
            'product_name': row['product_name'][:70],
            'category': str(row['category']).split('|')[0],
            'rating': row['rating'],
            'hybrid_score': round(float(hybrid[i]), 4),
            'cb_score': round(float(cb_norm[i]), 4),
            'cf_score': round(float(cf_norm[i]), 4),
        })
    return pd.DataFrame(recs)

print('Fungsi hybrid_recommend() siap ✓')

## 7. Demo & Evaluasi

In [ ]:
# ── 7.1 Demo rekomendasi ──────────────────────────────────────
sample_pid = prod_df['product_id'].iloc[0]
sample_name = prod_df['product_name'].iloc[0][:60]
print(f'Produk query: {sample_name}')
print('='*70)

recs = hybrid_recommend(sample_pid, top_n=5)
display(recs)

In [ ]:
# ── 7.2 Bandingkan CB-only vs Hybrid ─────────────────────────
def cb_only_recommend(product_id, top_n=5):
    if product_id not in prod_idx: return pd.DataFrame()
    idx = prod_idx[product_id]
    scores = cb_sim[idx].copy()
    scores[idx] = -1
    top_idx = np.argsort(scores)[::-1][:top_n]
    return [prod_ids[i] for i in top_idx]

cb_recs = cb_only_recommend(sample_pid)
hybrid_recs = hybrid_recommend(sample_pid)['product_id'].tolist()

overlap = len(set(cb_recs) & set(hybrid_recs))
print(f'Overlap CB vs Hybrid top-5: {overlap}/5')
print(f'CF contribution mendorong {5-overlap} produk berbeda masuk top-5')

In [ ]:
# ── 7.3 Coverage & statistik ──────────────────────────────────
import random
sample_pids = random.sample(prod_ids, min(100, len(prod_ids)))

all_rec_scores = []
for pid in sample_pids:
    recs = hybrid_recommend(pid, top_n=5)
    if len(recs):
        all_rec_scores.extend(recs['hybrid_score'].tolist())

all_rec_scores = np.array(all_rec_scores)
print(f'Evaluasi pada {len(sample_pids)} produk sample:')
print(f'  Mean hybrid score : {all_rec_scores.mean():.4f}')
print(f'  Std hybrid score  : {all_rec_scores.std():.4f}')
print(f'  Min score         : {all_rec_scores.min():.4f}')
print(f'  Max score         : {all_rec_scores.max():.4f}')

## 8. Ringkasan Pipeline

| Komponen | Detail |
|----------|--------|
| **Dataset** | 1.464 baris → 1.350 produk unik |
| **Explode** | ~9.042 user unik setelah explode `user_id` |
| **Text** | `about_product` + aggregated `review_content` |
| **Preprocessing** | lowercase, hapus URL, hapus tanda baca |
| **TF-IDF** | max_features=5000, ngram=(1,2), min_df=2 |
| **CB Similarity** | Cosine similarity → matrix 1350×1350 |
| **CF** | User-item matrix → item-item cosine similarity |
| **Hybrid** | α=0.4×CF + 0.6×CB (dapat di-tune) |
| **Output** | Top-5 rekomendasi per produk |

### Catatan Tuning
- `ALPHA` bisa diubah: `0.0` = pure CB, `1.0` = pure CF
- `max_features` TF-IDF bisa dinaikkan untuk representasi lebih kaya
- Bisa ditambah stopwords Bahasa Indonesia jika ada review dalam Bahasa Indonesia